# exp02 - PharmaSales weekly: perbandingan di bawah protokol tunggal

Notebook ini **menggantikan** perbandingan lintas-notebook pada revisi sebelumnya
(`our_study_pharma_weekly.ipynb` vs `rathipriya_pharma_daily.ipynb`), yang oleh kedua
reviewer IJIES dinilai tidak sah karena setiap model dijalankan dengan preprocessing,
pembagian data, dan protokol tuning yang berbeda.

## Kontrak eksperimen

Semua angka di notebook ini dihasilkan di bawah satu kontrak yang dikodekan di
`src/experiments/protocol.py` (satu-satunya sumber kebenaran; notebook lain memakai
modul yang sama sehingga protokolnya tidak mungkin menyimpang):

| Aspek | Ketentuan |
|---|---|
| Seed | `SEED = 42`, dipasang ke `random`, NumPy, `PYTHONHASHSEED`; setiap estimator stokastik menerima `random_state=SEED` melalui satu konstruktor `make_xgb()` |
| Split | Kronologis 70 / 15 / 15 (train / validation / test), tanpa pengacakan, **identik untuk semua model dan semua set fitur** |
| Tuning | Hyperparameter dipilih **hanya** dari RMSE validation. Test split disentuh **satu kali** oleh model final |
| Refit | Model final di-refit pada train+val dengan konfigurasi terbaik dari validation |
| Fitur | Set fitur adalah **faktor eksperimen**: `A_lag1` (protokol referensi) dan `B_rich` (lag terpilih + rolling mean) - setiap model dijalankan pada keduanya |
| Seleksi lag | argmax PACF dihitung **hanya pada blok training**; aturan alternatif diuji sebagai ablasi tersendiri |
| Penskalaan | Scaler di-fit ulang pada blok training aktif saja, tidak pernah pada test |

## Perbedaan terhadap pipeline lama (dan alasannya)

1. **Baseline dan model usulan memakai fitur serta split yang sama.** Sebelumnya baris
   "Reference" pada Tabel 2 hanya diberi `lag_1` dengan split 70/15/15, sedangkan model
   usulan memakai 2-16 fitur dengan split 60/20/20. Selisih yang dilaporkan karena itu
   mencampur efek model dengan efek preprocessing.
2. **Tidak ada grid search pada test set.** Cabang "Our Preprocessing" pada notebook
   baseline memilih hyperparameter dengan menilai test split (80/20 tanpa validation).
3. **PACF dihitung pada blok training saja.** Notebook lama menghitung ACF/PACF pada
   seluruh deret termasuk test - kebocoran halus pada tahap desain fitur.
4. **Grid bandwidth kernel diperlebar.** Pada grid lama (maksimum sigma = 5) optimum
   validation jatuh persis di batas atas untuk sebagian besar kategori, artinya grid
   memotong ruang pencarian dan secara sistematis merugikan GRNN/P_NN.
5. **Baseline naif ditambahkan.** Klaim keunggulan tanpa pembanding naif tidak dapat
   dinilai; Naive dan Seasonal Naive (s=52) kini dilaporkan di setiap tabel.
6. **Uji Diebold-Mariano dilaporkan** agar selisih RMSE dapat dinilai signifikansinya,
   bukan sekadar dibandingkan angkanya.

In [1]:
import sys, os, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiments import protocol as P

P.set_global_seed()

GRANULARITY      = "weekly"
DATA_PATH        = "../data/raw/pharma-sales/salesweekly.csv"
SEASONAL_PERIOD  = 52
EXPERIMENT       = "exp02_pharma_weekly"
CATEGORIES       = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

data = pd.read_csv(DATA_PATH)
print("Lingkungan:", P.environment_stamp())
print("Baris data mentah:", len(data))
data.head()

Lingkungan: {'python': '3.10.11', 'platform': 'Windows-10-10.0.22621-SP0', 'numpy': '2.2.6', 'pandas': '2.3.3', 'seed': '42', 'sklearn': '1.7.2', 'xgboost': '3.2.0', 'statsmodels': '0.14.6'}
Baris data mentah: 302


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,1/5/2014,14.00,11.67,21.3,185.95,41.0,0.0,32.0,7.0
1,1/12/2014,29.33,12.68,37.9,190.70,88.0,5.0,21.0,7.2
2,1/19/2014,30.67,26.34,45.9,218.40,80.0,8.0,29.0,12.0
3,1/26/2014,34.00,32.37,31.5,179.60,80.0,8.0,23.0,10.0
4,2/2/2014,31.02,23.35,20.7,159.88,84.0,12.0,29.0,12.0


## 1. Stabilitas aturan pemilihan lag

Nilai *k* (jumlah lag) menentukan seluruh set fitur `B_rich`. Notebook lama memakai
**dua aturan berbeda** untuk dua pipeline yang seharusnya dibandingkan: `our_study`
memakai argmax PACF, `rathipriya` (cabang "Our Preprocessing") memakai argmax ACF, dan
keduanya menghitung statistiknya pada deret penuh termasuk test.

Tabel di bawah menunjukkan bahwa *k* berubah drastis hanya karena pilihan tersebut.
Ini bukan detail teknis: bila *k* tidak stabil, kontribusi tahap rekayasa fitur pada
metode usulan juga tidak stabil, dan itu harus dilaporkan.

In [2]:
lag_table = []
for category in CATEGORIES:
    y = data[category].to_numpy(dtype=float)
    row = {"Category": category}
    for rule in ["pacf_train", "pacf_full", "acf_train", "acf_full", "pacf_significant"]:
        row[rule] = P.select_lag(y, rule=rule)
    lag_table.append(row)

lag_table = pd.DataFrame(lag_table)
print("k (jumlah lag) menurut aturan seleksi -- *_full memakai data test (bocor)")
lag_table

k (jumlah lag) menurut aturan seleksi -- *_full memakai data test (bocor)


,Category,pacf_train,pacf_full,acf_train,acf_full,pacf_significant
0,M01AB,2,2,9,2,12
1,M01AE,1,1,1,1,4
2,N02BA,1,1,1,1,25
3,N02BE,1,1,1,1,21
4,N05B,1,1,1,1,24
5,N05C,10,1,10,10,10
6,R03,1,1,1,1,21
7,R06,1,1,1,1,26


## 2. Desain split

Split dicetak secara eksplisit (tanggal awal/akhir setiap blok, ukuran setiap blok)
karena reviewer secara khusus meminta tanggal train/test yang hilang dari naskah.
Perhatikan bahwa `A_lag1` dan `B_rich` berbagi **baris dan tanggal yang persis sama**:
kedua set fitur dibangun dari kerangka yang sama dengan *k* yang sama, sehingga
satu-satunya yang berbeda adalah kolom fiturnya.

In [3]:
datasets = {}
split_rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        d = P.build_pharma_dataset(data, category, feature_set,
                                   seasonal_period=SEASONAL_PERIOD,
                                   lag_rule="pacf_train")
        datasets[(category, feature_set)] = d
        split_rows.append({**d.describe(), "features": ", ".join(d.feature_names[:4]) +
                           (" ..." if len(d.feature_names) > 4 else "")})

split_table = pd.DataFrame(split_rows)
split_table.to_csv(f"../results/{EXPERIMENT}_splits.csv", index=False)
split_table

,category,feature_set,lag_rule,n_features,n_lags,n_train,n_val,n_test,train_start,train_end,val_start,val_end,test_start,test_end,features
0,M01AB,A_lag1,pacf_train,1,2,210,45,45,2014-01-19,2018-01-21,2018-01-28,2018-12-02,2018-12-09,2019-10-13,lag_1
1,M01AB,B_rich,pacf_train,3,2,210,45,45,2014-01-19,2018-01-21,2018-01-28,2018-12-02,2018-12-09,2019-10-13,"lag_1, lag_2, rolling_mean_2"
2,M01AE,A_lag1,pacf_train,1,1,210,45,46,2014-01-12,2018-01-14,2018-01-21,2018-11-25,2018-12-02,2019-10-13,lag_1
3,M01AE,B_rich,pacf_train,1,1,210,45,46,2014-01-12,2018-01-14,2018-01-21,2018-11-25,2018-12-02,2019-10-13,lag_1
4,N02BA,A_lag1,pacf_train,1,1,210,45,46,2014-01-12,2018-01-14,2018-01-21,2018-11-25,2018-12-02,2019-10-13,lag_1
5,N02BA,B_rich,pacf_train,1,1,210,45,46,2014-01-12,2018-01-14,2018-01-21,2018-11-25,2018-12-02,2019-10-13,lag_1
6,N02BE,A_lag1,pacf_train,1,1,210,45,46,2014-01-12,2018-01-14,2018-01-21,2018-11-25,2018-12-02,2019-10-13,lag_1
7,N02BE,B_rich,pacf_train,1,1,210,45,46,2014-01-12,2018-01-14,2018-01-21,2018-11-25,2018-12-02,2019-10-13,lag_1
8,N05B,A_lag1,pacf_train,1,1,210,45,46,2014-01-12,2018-01-14,2018-01-21,2018-11-25,2018-12-02,2019-10-13,lag_1
9,N05B,B_rich,pacf_train,1,1,210,45,46,2014-01-12,2018-01-14,2018-01-21,2018-11-25,2018-12-02,2019-10-13,lag_1


## 3. Menjalankan seluruh model

Setiap model dieksekusi lewat `P.run_model`, yang menegakkan kontrak tuning:
grid dievaluasi pada validation, konfigurasi terbaik di-refit pada train+val, lalu
test diprediksi tepat satu kali. Tidak ada jalur kode di mana test dapat memengaruhi
pemilihan hyperparameter.

`XGBoost`, `LR+XGB (rata-rata)` dan `LR-XGB (residual)` memakai grid yang sama
(`GRID_XGB_PHARMA`, 12 konfigurasi) agar anggaran tuningnya setara.

In [4]:
MODELS = [
    # (nama, fungsi fit_predict, grid, jenis scaler)
    ("LR",                  P.fp_linear_regression, None,               None),
    ("GRNN",                P.fp_grnn,              P.GRID_GRNN,        "standard"),
    ("P_NN",                P.fp_pnn,               P.GRID_PNN,         "standard"),
    ("RBFNN",               P.fp_rbfnn,             P.GRID_RBFNN,       "standard"),
    ("XGBoost",             P.fp_xgboost,           P.GRID_XGB_PHARMA,  None),
    ("LR+XGB (average)",    P.fp_lr_xgb_average,    P.GRID_XGB_PHARMA,  None),
    ("LR-XGB (residual)",   P.fp_lr_xgb_residual,   P.GRID_XGB_PHARMA,  None),
]

rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        d = datasets[(category, feature_set)]
        rows += P.naive_rows(d)
        for name, fn, grid, scaler in MODELS:
            rows.append(P.run_model(name, fn, d, grid, scaler_kind=scaler))
    print(f"selesai: {category}", flush=True)

# ARIMA(5,1,0) bersifat univariat: tidak bergantung pada set fitur, dijalankan sekali
# per kategori dan dilaporkan pada kedua set fitur dengan penanda eksplisit.
for category in CATEGORIES:
    d = datasets[(category, P.FEATURE_SET_A)]
    try:
        rows.append(P.run_model("ARIMA(5,1,0)", P.fp_arima, d,
                                {"order": [(5, 1, 0)]},
                                extra={"note": "univariat; tidak memakai fitur"}))
    except Exception as exc:
        print(f"ARIMA gagal untuk {category}: {exc}")

results = P.save_results(rows, EXPERIMENT)
print(f"{len(results)} baris hasil ditulis ke ../results/{EXPERIMENT}.csv")
results.head()

selesai: M01AB
selesai: M01AE
selesai: N02BA
selesai: N02BE
selesai: N05B
selesai: N05C
selesai: R03
selesai: R06
152 baris hasil ditulis ke ../results/exp02_pharma_weekly.csv


,model,category,feature_set,lag_rule,n_features,n_lags,n_train,n_val,n_test,train_start,train_end,val_start,val_end,test_start,test_end,params,n_grid,scaler,seed,val_MSE,val_RMSE,val_MAE,val_R2,val_nRMSE,val_SMAPE,val_RMSPE,test_MSE,test_RMSE,test_MAE,test_R2,test_nRMSE,test_SMAPE,test_RMSPE,runtime_s,note
0,Naive,M01AB,A_lag1,pacf_train,1,2,210,45,45,2014-01-19,2018-01-21,2018-01-28,2018-12-02,2018-12-09,2019-10-13,{},0,none,42,116.877716,10.811000,8.296889,-1.378380,0.313882,23.329639,0.317666,119.258058,10.920534,8.319556,-0.616483,0.296967,22.935915,0.660681,0.000,NaN
1,SeasonalNaive(s=52),M01AB,A_lag1,pacf_train,1,2,210,45,45,2014-01-19,2018-01-21,2018-01-28,2018-12-02,2018-12-09,2019-10-13,{},0,none,42,129.603968,11.384374,8.507685,-1.637350,0.330529,24.061894,0.386506,115.758531,10.759114,9.071778,-0.569049,0.292577,26.511852,0.554395,0.000,NaN
2,LR,M01AB,A_lag1,pacf_train,1,2,210,45,45,2014-01-19,2018-01-21,2018-01-28,2018-12-02,2018-12-09,2019-10-13,{},1,none,42,58.258099,7.632699,6.062765,-0.185512,0.221605,17.337584,0.228824,76.025662,8.719270,6.422270,-0.030490,0.237107,18.518088,0.581465,0.171,NaN
3,GRNN,M01AB,A_lag1,pacf_train,1,2,210,45,45,2014-01-19,2018-01-21,2018-01-28,2018-12-02,2018-12-09,2019-10-13,"{""sigma"": 50.0}",16,standard,42,49.469469,7.033454,5.716111,-0.006669,0.204206,16.401921,0.217574,77.236569,8.788434,6.482630,-0.046903,0.238988,18.731316,0.568597,0.013,NaN
4,P_NN,M01AB,A_lag1,pacf_train,1,2,210,45,45,2014-01-19,2018-01-21,2018-01-28,2018-12-02,2018-12-09,2019-10-13,"{""sigma"": 1.0}",16,standard,42,50.057264,7.075116,5.160609,-0.018630,0.205416,14.732584,0.196024,85.331737,9.237518,7.150462,-0.156629,0.251200,20.891215,0.535113,0.016,NaN


## 4. Tabel utama - RMSE test per kategori x set fitur

Ini adalah tabel yang menggantikan Tabel 2 pada naskah. Semua sel dihasilkan dari
satu proses, satu split, satu protokol tuning, dan satu seed.

In [5]:
pivot = results.pivot_table(index=["category", "feature_set"], columns="model",
                            values="test_RMSE")
order = ["Naive", f"SeasonalNaive(s={SEASONAL_PERIOD})", "ARIMA(5,1,0)", "LR",
         "GRNN", "P_NN", "RBFNN", "XGBoost", "LR+XGB (average)", "LR-XGB (residual)"]
pivot = pivot[[c for c in order if c in pivot.columns]]
display(pivot.round(4))

winner = pivot.idxmin(axis=1).rename("model terbaik (RMSE test)")
print("\nModel terbaik per (kategori, set fitur):")
print(winner.to_string())
print("\nBerapa kali metode usulan menang:",
      int((winner == "LR-XGB (residual)").sum()), "dari", len(winner))

model                   Naive  SeasonalNaive(s=52)  ARIMA(5,1,0)       LR     GRNN     P_NN    RBFNN  XGBoost  LR+XGB (average)  LR-XGB (residual)
category feature_set                                                                                                                              
M01AB    A_lag1       10.9205              10.7591        8.8008   8.7193   8.7884   9.2375   8.7108   8.8864            8.6943             8.8407
         B_rich       10.9205              10.7591           NaN   8.6713   8.7781   9.1831   8.6004   9.0576            8.8860             9.1217
M01AE    A_lag1        9.9500              10.5276       10.2251   9.2275  10.2813  10.6194  10.2508   9.6425            9.3917             9.4054
         B_rich        9.9500              10.5276           NaN   9.2275  10.2813  10.6194  10.2508   9.6425            9.3917             9.4054
N02BA    A_lag1        6.2395               7.2071        6.5336   6.8662   7.1841   6.2623   6.7827   7.1617            6.9652             7.1327
         B_rich        6.2395               7.2071           NaN   6.8662   7.1841   6.2623   6.7827   7.1617            6.9652             7.1327
N02BE    A_lag1       53.5960              61.8659       85.9269  51.3467  50.3296  55.3193  51.1346  48.7267           49.3749            49.5183
         B_rich       53.5960              61.8659           NaN  51.3467  50.3296  55.3193  51.1346  48.7267           49.3749            49.5183
N05B     A_lag1       15.4933              21.3796       12.3257  12.9702  12.7175  12.9179  12.5360  13.6580           13.1263            13.2927
         B_rich       15.4933              21.3796           NaN  12.9702  12.7175  12.9179  12.5360  13.6580           13.1263            13.2927
N05C     A_lag1        4.0332               4.8831        2.8170   2.9551   2.9744   3.4476   3.0717   3.1122            3.0160             3.1126
         B_rich        4.0332               4.8831           NaN   2.9410   2.8868   3.3635   2.9331   2.9094            3.0543             2.9432
R03      A_lag1       29.1849              33.6463       43.1389  26.7117  26.9553  27.1508  27.8197  26.7305           26.3901            26.4546
         B_rich       29.1849              33.6463           NaN  26.7117  26.9553  27.1508  27.8197  26.7305           26.3901            26.4546
R06      A_lag1        9.4134              12.2579       19.8097   8.9271   9.3817  11.2203   8.8980   9.8346            9.2592             9.7152
         B_rich        9.4134              12.2579           NaN   8.9271   9.3817  11.2203   8.8980   9.8346            9.2592             9.7152


Model terbaik per (kategori, set fitur):
category  feature_set
M01AB     A_lag1         LR+XGB (average)
          B_rich                    RBFNN
M01AE     A_lag1                       LR
          B_rich                       LR
N02BA     A_lag1                    Naive
          B_rich                    Naive
N02BE     A_lag1                  XGBoost
          B_rich                  XGBoost
N05B      A_lag1             ARIMA(5,1,0)
          B_rich                    RBFNN
N05C      A_lag1             ARIMA(5,1,0)
          B_rich                     GRNN
R03       A_lag1         LR+XGB (average)
          B_rich         LR+XGB (average)
R06       A_lag1                    RBFNN
          B_rich                    RBFNN

Berapa kali metode usulan menang: 0 dari 16


## 5. Apakah selisihnya signifikan? (Diebold-Mariano)

Selisih RMSE beberapa persen pada n_test kecil tidak otomatis berarti model lebih baik.
Uji DM di bawah membandingkan metode usulan dengan (a) baseline terbaik non-hibrida
pada kondisi yang sama dan (b) Seasonal Naive. Nilai DM negatif berarti metode usulan
lebih akurat; `p_value` di atas 0.05 berarti selisihnya tidak dapat dibedakan dari nol.

In [6]:
by_key = {(r["category"], r["feature_set"], r["model"]): r for r in rows}
dm_rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        prop = by_key.get((category, feature_set, "LR-XGB (residual)"))
        if prop is None:
            continue
        d = datasets[(category, feature_set)]
        competitors = [m for m in ["LR", "GRNN", "P_NN", "RBFNN", "XGBoost",
                                   f"SeasonalNaive(s={SEASONAL_PERIOD})"]
                       if (category, feature_set, m) in by_key]
        best = min(competitors,
                   key=lambda m: by_key[(category, feature_set, m)]["test_RMSE"])
        for other in {best, f"SeasonalNaive(s={SEASONAL_PERIOD})"}:
            ref = by_key[(category, feature_set, other)]
            test = P.diebold_mariano(d.y_test, prop["_test_pred"], ref["_test_pred"])
            dm_rows.append({
                "category": category, "feature_set": feature_set,
                "pembanding": other,
                "RMSE usulan": round(prop["test_RMSE"], 4),
                "RMSE pembanding": round(ref["test_RMSE"], 4),
                "DM": round(test["DM"], 3) if test["DM"] == test["DM"] else None,
                "p_value": round(test["p_value"], 4) if test["p_value"] == test["p_value"] else None,
                "signifikan (a=0.05)": (test["p_value"] < 0.05) if test["p_value"] == test["p_value"] else None,
            })

dm_table = pd.DataFrame(dm_rows).sort_values(["category", "feature_set", "pembanding"])
dm_table.to_csv(f"../results/{EXPERIMENT}_dm_test.csv", index=False)
dm_table

,category,feature_set,pembanding,RMSE usulan,RMSE pembanding,DM,p_value,signifikan (a=0.05)
0,M01AB,A_lag1,RBFNN,8.8407,8.7108,0.379,0.7064,False
1,M01AB,A_lag1,SeasonalNaive(s=52),8.8407,10.7591,-1.907,0.0630,False
2,M01AB,B_rich,RBFNN,9.1217,8.6004,1.517,0.1363,False
3,M01AB,B_rich,SeasonalNaive(s=52),9.1217,10.7591,-1.407,0.1666,False
4,M01AE,A_lag1,LR,9.4054,9.2275,0.940,0.3522,False
5,M01AE,A_lag1,SeasonalNaive(s=52),9.4054,10.5276,-1.358,0.1812,False
6,M01AE,B_rich,LR,9.4054,9.2275,0.940,0.3522,False
7,M01AE,B_rich,SeasonalNaive(s=52),9.4054,10.5276,-1.358,0.1812,False
8,N02BA,A_lag1,P_NN,7.1327,6.2623,1.922,0.0609,False
9,N02BA,A_lag1,SeasonalNaive(s=52),7.1327,7.2071,-0.095,0.9248,False


## 6. Ablasi aturan seleksi lag

Faktor ketiga: apakah kesimpulan berubah bila *k* dipilih dengan aturan lain?
Bagian ini menjalankan ulang metode usulan dan LR di bawah lima aturan seleksi lag.
Bila peringkat model berubah-ubah antar aturan, klaim keunggulan harus dilemahkan
secara eksplisit di naskah.

In [7]:
ablation_rows = []
for rule in ["pacf_train", "pacf_full", "acf_train", "acf_full", "pacf_significant"]:
    for category in CATEGORIES:
        d = P.build_pharma_dataset(data, category, P.FEATURE_SET_B,
                                   seasonal_period=SEASONAL_PERIOD, lag_rule=rule)
        ablation_rows.append(P.run_model("LR", P.fp_linear_regression, d))
        ablation_rows.append(P.run_model("LR-XGB (residual)", P.fp_lr_xgb_residual,
                                         d, P.GRID_XGB_PHARMA))
    print("selesai aturan:", rule, flush=True)

ablation = P.save_results(ablation_rows, f"{EXPERIMENT}_lag_ablation")
ablation.pivot_table(index=["category", "lag_rule"], columns="model",
                     values="test_RMSE").round(4)

selesai aturan: pacf_train
selesai aturan: pacf_full
selesai aturan: acf_train
selesai aturan: acf_full
selesai aturan: pacf_significant


model                           LR  LR-XGB (residual)
category lag_rule                                    
M01AB    acf_full           8.6713             9.1217
         acf_train          9.0544             9.5037
         pacf_full          8.6713             9.1217
         pacf_significant   8.7425             9.2933
         pacf_train         8.6713             9.1217
M01AE    acf_full           9.2275             9.4054
         acf_train          9.2275             9.4054
         pacf_full          9.2275             9.4054
         pacf_significant   9.4718            10.0883
         pacf_train         9.2275             9.4054
N02BA    acf_full           6.8662             7.1327
         acf_train          6.8662             7.1327
         pacf_full          6.8662             7.1327
         pacf_significant   6.0924             6.2496
         pacf_train         6.8662             7.1327
N02BE    acf_full          51.3467            49.5183
         acf_train         51.3467            49.5183
         pacf_full         51.3467            49.5183
         pacf_significant  58.0199            60.5167
         pacf_train        51.3467            49.5183
N05B     acf_full          12.9702            13.2927
         acf_train         12.9702            13.2927
         pacf_full         12.9702            13.2927
         pacf_significant  13.1388            14.2622
         pacf_train        12.9702            13.2927
N05C     acf_full           2.9410             2.9432
         acf_train          2.9410             2.9432
         pacf_full          2.9155             3.0893
         pacf_significant   2.9410             2.9432
         pacf_train         2.9410             2.9432
R03      acf_full          26.7117            26.4546
         acf_train         26.7117            26.4546
         pacf_full         26.7117            26.4546
         pacf_significant  28.3023            29.6693
         pacf_train        26.7117            26.4546
R06      acf_full           8.9271             9.7152
         acf_train          8.9271             9.7152
         pacf_full          8.9271             9.7152
         pacf_significant   8.9556             8.9821
         pacf_train         8.9271             9.7152

## 7. Pemeriksaan determinisme

Klaim reproduktifitas harus diuji, bukan dinyatakan. Sel di bawah menjalankan ulang
metode usulan untuk seluruh kategori pada proses yang sama dan membandingkan prediksi
bit-per-bit dengan hasil pertama.

In [8]:
P.set_global_seed()
identical = True
for category in CATEGORIES:
    d = datasets[(category, P.FEATURE_SET_B)]
    again = P.run_model("LR-XGB (residual)", P.fp_lr_xgb_residual, d, P.GRID_XGB_PHARMA)
    first = by_key[(category, P.FEATURE_SET_B, "LR-XGB (residual)")]
    same = np.array_equal(again["_test_pred"], first["_test_pred"])
    identical &= same
    print(f"{category:6s} prediksi identik: {same}  | RMSE {again['test_RMSE']:.6f} "
          f"vs {first['test_RMSE']:.6f}")

print("\nSEMUA IDENTIK:", identical)

M01AB  prediksi identik: True  | RMSE 9.121704 vs 9.121704
M01AE  prediksi identik: True  | RMSE 9.405358 vs 9.405358
N02BA  prediksi identik: True  | RMSE 7.132658 vs 7.132658
N02BE  prediksi identik: True  | RMSE 49.518292 vs 49.518292
N05B   prediksi identik: True  | RMSE 13.292703 vs 13.292703
N05C   prediksi identik: True  | RMSE 2.943187 vs 2.943187
R03    prediksi identik: True  | RMSE 26.454569 vs 26.454569
R06    prediksi identik: True  | RMSE 9.715241 vs 9.715241

SEMUA IDENTIK: True


## 8. Catatan untuk naskah

* Semua angka di notebook ini berada pada **skala asli** unit penjualan weekly; tidak
  ada metrik yang dilaporkan pada skala transformasi tanpa penanda. Untuk setiap baris,
  `RMSE = sqrt(MSE)` secara eksak menurut konstruksi (`compute_metrics`), sehingga
  ketidakkonsistenan RMSE/MSE yang ditemukan reviewer tidak dapat terulang.
* `results/{EXPERIMENT}.csv` berisi satu baris per (kategori, set fitur, model) lengkap
  dengan hyperparameter terpilih, metrik validation, metrik test, ukuran split, tanggal
  split, dan seed - format machine-readable yang diminta reviewer.
* Bila kolom `n_features` untuk `A_lag1` dan `B_rich` bernilai sama pada suatu kategori,
  itu berarti argmax PACF pada blok training bernilai 1 sehingga `rolling_mean_1`
  identik dengan `lag_1` dan dibuang sebagai kolom duplikat. Kondisi ini dilaporkan
  apa adanya: untuk kategori tersebut pipeline "kaya" memang berdegenerasi menjadi
  pipeline referensi.